## 1. See what currencies and dates we need

In [1]:
%%sql
-- Lists the distinct currencies in the clean orders.
-- Tells us which currencies need converting to EUR (EUR needs no conversion).
SELECT DISTINCT currency FROM orders_clean ORDER BY currency

StatementMeta(, 52309d1f-9d22-499a-bffb-a469af3fe7b2, 2, Finished, Available, Finished, True)

<Spark SQL result set with 2 rows and 1 fields>

In [1]:
%%sql
-- Shows the earliest/latest fx_reference_date and how many distinct dates.
-- Tells us the date range of exchange rates to pull.
-- a first date, a last date, and a count of distinct dates.
SELECT
  MIN(fx_reference_date) AS first_date,
  MAX(fx_reference_date) AS last_date,
  COUNT(DISTINCT fx_reference_date) AS distinct_dates
FROM orders_clean

StatementMeta(, f8e37aba-7700-4027-8445-68b426001a6a, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>


| Result | Meaning |
|--------|---------|
| `first_date` = 2026-08-23 | the earliest date any order points to |
| `last_date` = 2026-09-03 | the latest date any order points to |
| `distinct_dates` = 12 | there are only 12 different dates in total |

This defines the exact date range of exchange rates we need to pull from frankfurter.dev
(`start = first_date`, `end = last_date`). It also shows the job is small (12 dates), and that
the last date is in the future — a hint that some dates won't have real market rates yet, so
we'll carry forward the most recent known rate for those.

## 2. Pull the exchange rates from frankfurter.dev

For each date we need, we ask the API for the EUR->RON rate. The API is smart: for a weekend it
returns the nearest previous business day's rate, and for a future date it returns the latest
known rate. So we always get the "most appropriate rate for the date" without extra logic.

In [2]:
import requests  # to call the exchange-rate API

# 1. get the list of dates we need, from the clean orders
dates = [str(r[0]) for r in spark.sql(
    "SELECT DISTINCT fx_reference_date FROM orders_clean ORDER BY fx_reference_date"
).collect()]


# 2. for each date, ask the API for the EUR->RON rate
#    (weekends -> nearest business day; future dates -> latest rate; handled by the API)
rows = []
for d in dates:
    resp = requests.get(f"https://api.frankfurter.dev/v1/{d}?to=RON").json()
    ron_rate = resp["rates"]["RON"]
    rows.append((d, "EUR", 1.0))        # EUR is the base currency, always 1
    rows.append((d, "RON", ron_rate))   # the RON rate for this date


# 3. save as the fx_rates table (fx_date stored as a real date so it joins cleanly later)
df = spark.createDataFrame(rows, ["fx_date", "currency", "rate_to_eur"])
df = df.withColumn("fx_date", df.fx_date.cast("date"))
df.write.mode("overwrite").saveAsTable("fx_rates")
print("Saved fx_rates:", len(rows), "rows")

StatementMeta(, f8e37aba-7700-4027-8445-68b426001a6a, 4, Finished, Available, Finished, False)

Saved fx_rates: 24 rows


In [3]:
%%sql
-- Shows the full fx_rates table.
-- one EUR (=1.0) and one RON row per date -> 24 rows (12 dates x 2 currencies).
SELECT * FROM fx_rates ORDER BY fx_date, currency

StatementMeta(, f8e37aba-7700-4027-8445-68b426001a6a, 5, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 3 fields>

**Results observation:**

2026-08-23 (Sunday) → 5.2563 (Friday's rate)

2026-08-24 → 5.2504 (real Monday)

2026-08-25 (today) → 5.2537

2026-08-26 onward (future) → 5.2537 (today's rate carried forward — this will update as those dates arrive)